In [1]:
!pip install -q pyspark

In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

root = "/content/drive/MyDrive/trialwatch"
print("trialwatch contents after move:")
for item in sorted(os.listdir(root)):
    print(repr(item))

Mounted at /content/drive
trialwatch contents after move:
'compliance_status_chart.png'
'overdue_by_sponsor_group.png'


In [9]:
# =========================
# STEP 2: SET UP SPARK
# =========================

# Install Spark
!pip install -q pyspark

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Imports
import os
import zipfile
from pyspark.sql import SparkSession

# -------------------------
# 1. SET YOUR AACT PATH
# -------------------------
AACT_PATH = "/content/drive/MyDrive/AACT data flattext"

print("AACT path exists:", os.path.exists(AACT_PATH))
print("AACT path:", AACT_PATH)

if not os.path.exists(AACT_PATH):
    raise FileNotFoundError(f"Folder not found: {AACT_PATH}")

print("\nContents of AACT folder:")
for f in sorted(os.listdir(AACT_PATH))[:50]:
    print(" -", repr(f))

# -------------------------
# 2. IF ONLY ZIP EXISTS, UNZIP IT
# -------------------------
zip_files = [f for f in os.listdir(AACT_PATH) if f.lower().endswith(".zip")]
txt_files = [f for f in os.listdir(AACT_PATH) if f.lower().endswith(".txt")]

if len(txt_files) == 0 and len(zip_files) > 0:
    zip_path = os.path.join(AACT_PATH, zip_files[0])
    extract_path = os.path.join(AACT_PATH, "unzipped")
    os.makedirs(extract_path, exist_ok=True)

    print("\nZip found, extracting now...")
    print("Zip file:", zip_path)
    print("Extracting to:", extract_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    AACT_PATH = extract_path

    print("\nUpdated AACT_PATH after extraction:")
    print(AACT_PATH)

    print("\nContents of extracted folder:")
    for f in sorted(os.listdir(AACT_PATH))[:50]:
        print(" -", repr(f))

elif len(txt_files) > 0:
    print("\nTXT files already present. No unzip needed.")

else:
    print("\nNo zip and no txt files found.")
    print("Please verify the folder contents manually.")

# -------------------------
# 3. START SPARK SESSION
# -------------------------
spark = SparkSession.builder \
    .appName("TrialWatch-AACT-Step2") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("\nSpark started successfully.")
print("Spark version:", spark.version)

# -------------------------
# 4. HELPER FUNCTION
# -------------------------
def read_aact_table(table_name):
    file_path = os.path.join(AACT_PATH, f"{table_name}.txt")
    print(f"\nTrying to read: {file_path}")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    df = spark.read \
        .option("header", True) \
        .option("sep", "|") \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("multiLine", False) \
        .csv(file_path)

    return df

# -------------------------
# 5. LOAD STUDIES TABLE
# -------------------------
studies_df = read_aact_table("studies")

print("\nLoaded studies table successfully.")
print("Row count:", studies_df.count())
print("Column count:", len(studies_df.columns))

print("\nSchema:")
studies_df.printSchema()

print("\nSample rows:")
studies_df.show(5, truncate=False)

# -------------------------
# 6. OPTIONAL: LOAD OTHER TABLES
# -------------------------
optional_tables = ["sponsors", "designs", "interventions", "browse_conditions"]

loaded = {}
for table_name in optional_tables:
    try:
        df = read_aact_table(table_name)
        loaded[table_name] = df
        print(f"\nLoaded {table_name} successfully.")
        print("Rows:", df.count())
        print("Columns:", len(df.columns))
    except Exception as e:
        print(f"\nCould not load {table_name}: {e}")

print("\nStep 2 complete.")

Mounted at /content/drive
AACT path exists: True
AACT path: /content/drive/MyDrive/AACT data flattext

Contents of AACT folder:
 - 'AACT data flattext'
 - 'baseline_counts.txt'
 - 'baseline_measurements.txt'
 - 'brief_summaries.txt'
 - 'browse_conditions.txt'
 - 'browse_interventions.txt'
 - 'calculated_values.txt'
 - 'central_contacts.txt'
 - 'conditions.txt'
 - 'countries.txt'
 - 'design_group_interventions.txt'
 - 'design_groups.txt'
 - 'design_outcomes.txt'
 - 'designs.txt'
 - 'detailed_descriptions.txt'
 - 'documents.txt'
 - 'drop_withdrawals.txt'
 - 'eligibilities.txt'
 - 'facilities.txt'
 - 'facility_contacts.txt'
 - 'facility_investigators.txt'
 - 'id_information.txt'
 - 'intervention_other_names.txt'
 - 'interventions.txt'
 - 'ipd_information_types.txt'
 - 'keywords.txt'
 - 'links.txt'
 - 'milestones.txt'
 - 'outcome_analyses.txt'
 - 'outcome_analysis_groups.txt'
 - 'outcome_counts.txt'
 - 'outcome_measurements.txt'
 - 'outcomes.txt'
 - 'overall_officials.txt'
 - 'participant_

In [10]:
# =========================
# STEP 3: BUILD TRIAL TABLE + COMPLIANCE METRICS
# =========================

from pyspark.sql.functions import (
    col, lit, when, coalesce, to_date, regexp_replace,
    datediff, current_date, expr, upper
)

# -------------------------
# 1. LOAD REQUIRED TABLES
# -------------------------
# studies_df should already exist from Step 2
# load supporting tables if available

def safe_load(table_name):
    try:
        df = read_aact_table(table_name)
        print(f"Loaded {table_name}")
        return df
    except Exception as e:
        print(f"Could not load {table_name}: {e}")
        return None

sponsors_df = safe_load("sponsors")
designs_df = safe_load("designs")

# -------------------------
# 2. INSPECT KEY COLUMNS
# -------------------------
print("\nSTUDIES COLUMNS:")
print(studies_df.columns)

if sponsors_df is not None:
    print("\nSPONSORS COLUMNS:")
    print(sponsors_df.columns)

if designs_df is not None:
    print("\nDESIGNS COLUMNS:")
    print(designs_df.columns)

# -------------------------
# 3. PICK LEAD SPONSOR ONLY
# -------------------------
# AACT sponsor table may have multiple rows per trial.
# We want the lead sponsor where possible.

if sponsors_df is not None:
    sponsor_cols = sponsors_df.columns

    sponsor_name_col = "name" if "name" in sponsor_cols else None
    sponsor_class_col = "agency_class" if "agency_class" in sponsor_cols else None
    sponsor_lead_col = "lead_or_collaborator" if "lead_or_collaborator" in sponsor_cols else None
    sponsor_nct_col = "nct_id" if "nct_id" in sponsor_cols else None

    if sponsor_nct_col and sponsor_name_col:
        lead_sponsors_df = sponsors_df

        if sponsor_lead_col:
            lead_sponsors_df = lead_sponsors_df.filter(
                upper(col(sponsor_lead_col)) == "LEAD"
            )

        lead_sponsors_df = lead_sponsors_df.select(
            col(sponsor_nct_col).alias("nctid"),
            col(sponsor_name_col).alias("orgname"),
            col(sponsor_class_col).alias("orgclass") if sponsor_class_col else lit(None).alias("orgclass")
        ).dropDuplicates(["nctid"])
    else:
        lead_sponsors_df = None
else:
    lead_sponsors_df = None

# -------------------------
# 4. GET DESIGN / PHASE DATA
# -------------------------
if designs_df is not None:
    design_cols = designs_df.columns

    design_nct_col = "nct_id" if "nct_id" in design_cols else None
    phase_col = "phase" if "phase" in design_cols else None

    if design_nct_col and phase_col:
        phase_df = designs_df.select(
            col(design_nct_col).alias("nctid"),
            col(phase_col).alias("phases")
        ).dropDuplicates(["nctid"])
    else:
        phase_df = None
else:
    phase_df = None

# -------------------------
# 5. BUILD BASE TRIAL TABLE
# -------------------------
study_cols = studies_df.columns

# Try common AACT study column names
nct_col = "nct_id" if "nct_id" in study_cols else None
study_type_col = "study_type" if "study_type" in study_cols else None
overall_status_col = "overall_status" if "overall_status" in study_cols else None
primary_completion_col = "primary_completion_date" if "primary_completion_date" in study_cols else None
results_submitted_col = "results_first_submitted_date" if "results_first_submitted_date" in study_cols else None
study_phase_col = "phase" if "phase" in study_cols else None

if not nct_col:
    raise ValueError("Could not find nct_id column in studies table.")

trial_df = studies_df.select(
    col(nct_col).alias("nctid"),
    col(study_type_col).alias("studytype") if study_type_col else lit(None).alias("studytype"),
    col(overall_status_col).alias("overallstatus") if overall_status_col else lit(None).alias("overallstatus"),
    col(primary_completion_col).alias("primarycompletiondate") if primary_completion_col else lit(None).alias("primarycompletiondate"),
    col(results_submitted_col).alias("resultsfirstsubmitdate") if results_submitted_col else lit(None).alias("resultsfirstsubmitdate"),
    col(study_phase_col).alias("study_phase") if study_phase_col else lit(None).alias("study_phase")
)

if lead_sponsors_df is not None:
    trial_df = trial_df.join(lead_sponsors_df, on="nctid", how="left")
else:
    trial_df = trial_df.withColumn("orgname", lit(None)).withColumn("orgclass", lit(None))

if phase_df is not None:
    trial_df = trial_df.join(phase_df, on="nctid", how="left")
else:
    trial_df = trial_df.withColumn("phases", lit(None))

# Use designs.phase if available, otherwise fall back to studies.phase
trial_df = trial_df.withColumn(
    "phases",
    coalesce(col("phases"), col("study_phase"))
).drop("study_phase")

print("\nBase trial table preview:")
trial_df.show(5, truncate=False)
print("Base trial row count:", trial_df.count())

# -------------------------
# 6. CLEAN DATE FIELDS
# -------------------------
# AACT may store partial dates; your old ETL normalized these into usable dates.
# Here we convert to proper date columns.

trial_df = trial_df.withColumn(
    "primarycompletiondate_clean",
    regexp_replace(col("primarycompletiondate"), r"^(\d{4}-\d{2})$", "\\1-01")
)

trial_df = trial_df.withColumn(
    "primarydt",
    to_date(col("primarycompletiondate_clean"))
)

trial_df = trial_df.withColumn(
    "resultsdt",
    to_date(col("resultsfirstsubmitdate"))
)

# -------------------------
# 7. FILTER TO ACT-LIKE TRIALS
# -------------------------
# Match your earlier logic: interventional + phase 2/3/4 + primary completion date present

act_df = trial_df.filter(
    (upper(col("studytype")) == "INTERVENTIONAL") &
    (
        upper(col("phases")).isin("PHASE2", "PHASE3", "PHASE4")
    ) &
    col("primarydt").isNotNull()
)

print("\nACT trial count:")
print(act_df.count())

# -------------------------
# 8. DERIVE SPONSOR GROUP
# -------------------------
# Your downstream trials collection included sponsorgroup.
# We'll map orgclass into a simple sponsor group.

act_df = act_df.withColumn(
    "sponsorgroup",
    when(upper(col("orgclass")) == "INDUSTRY", lit("INDUSTRY"))
    .when(upper(col("orgclass")).isin("NIH", "FED"), lit("GOVERNMENT"))
    .otherwise(lit("ACADEMICOROTHER"))
)

# -------------------------
# 9. COMPUTE DEADLINE + COMPLIANCE
# -------------------------
# Match the compliance schema your Mongo loader expects

act_df = act_df.withColumn(
    "deadline",
    expr("date_add(primarydt, 365)")
)

act_df = act_df.withColumn(
    "compliancestatusv2",
    when(col("resultsdt").isNull() & (current_date() > col("deadline")), lit("MISSING"))
    .when(col("resultsdt").isNull() & (current_date() <= col("deadline")), lit("NOTDUEYET"))
    .when(col("resultsdt") <= col("deadline"), lit("COMPLIANT"))
    .otherwise(lit("LATE"))
)

act_df = act_df.withColumn(
    "daysoverdue",
    when(col("compliancestatusv2") == "COMPLIANT", lit(0))
    .when(col("compliancestatusv2") == "NOTDUEYET", lit(0))
    .when(col("resultsdt").isNull(), datediff(current_date(), col("deadline")))
    .otherwise(datediff(col("resultsdt"), col("deadline")))
)

act_df = act_df.withColumn(
    "isoverdue",
    when(col("compliancestatusv2").isin("LATE", "MISSING"), lit(True)).otherwise(lit(False))
)

print("\nCompliance breakdown:")
act_df.groupBy("compliancestatusv2").count().show()

# -------------------------
# 10. BUILD FINAL OUTPUT TABLES
# -------------------------
trialsclean_df = act_df.select(
    "nctid",
    "orgname",
    "orgclass",
    "sponsorgroup",
    "studytype",
    "phases",
    "overallstatus",
    "primarycompletiondate",
    "resultsfirstsubmitdate",
    "primarydt",
    "resultsdt"
)

compliance_df = act_df.select(
    "nctid",
    "primarydt",
    "resultsdt",
    "deadline",
    "compliancestatusv2",
    "daysoverdue",
    "isoverdue"
)

print("\ntrialsclean preview:")
trialsclean_df.show(5, truncate=False)

print("\ncompliancemetrics preview:")
compliance_df.show(5, truncate=False)

# -------------------------
# 11. SAVE TO DRIVE
# -------------------------
TRIALS_OUT = "/content/drive/MyDrive/trialsclean_spark"
COMP_OUT = "/content/drive/MyDrive/compliancemetrics_spark"

trialsclean_df.coalesce(1).write.mode("overwrite").option("header", True).csv(TRIALS_OUT)
compliance_df.coalesce(1).write.mode("overwrite").option("header", True).csv(COMP_OUT)

print("\nSaved Spark outputs:")
print("trialsclean_spark ->", TRIALS_OUT)
print("compliancemetrics_spark ->", COMP_OUT)


Trying to read: /content/drive/MyDrive/AACT data flattext/sponsors.txt
Loaded sponsors

Trying to read: /content/drive/MyDrive/AACT data flattext/designs.txt
Loaded designs

STUDIES COLUMNS:
['nct_id', 'nlm_download_date_description', 'study_first_submitted_date', 'results_first_submitted_date', 'disposition_first_submitted_date', 'last_update_submitted_date', 'study_first_submitted_qc_date', 'study_first_posted_date', 'study_first_posted_date_type', 'results_first_submitted_qc_date', 'results_first_posted_date', 'results_first_posted_date_type', 'disposition_first_submitted_qc_date', 'disposition_first_posted_date', 'disposition_first_posted_date_type', 'last_update_submitted_qc_date', 'last_update_posted_date', 'last_update_posted_date_type', 'start_month_year', 'start_date_type', 'start_date', 'verification_month_year', 'verification_date', 'completion_month_year', 'completion_date_type', 'completion_date', 'primary_completion_month_year', 'primary_completion_date_type', 'primary_c

In [11]:
import os
import shutil

TRIALS_FOLDER = "/content/drive/MyDrive/trialsclean_spark"
COMP_FOLDER = "/content/drive/MyDrive/compliancemetrics_spark"

trial_part = [f for f in os.listdir(TRIALS_FOLDER) if f.startswith("part-") and f.endswith(".csv")][0]
comp_part = [f for f in os.listdir(COMP_FOLDER) if f.startswith("part-") and f.endswith(".csv")][0]

trial_src = os.path.join(TRIALS_FOLDER, trial_part)
comp_src = os.path.join(COMP_FOLDER, comp_part)

trial_dst = "/content/drive/MyDrive/trialsclean.csv"
comp_dst = "/content/drive/MyDrive/compliancemetrics.csv"

shutil.copy(trial_src, trial_dst)
shutil.copy(comp_src, comp_dst)

print("Saved:", trial_dst)
print("Saved:", comp_dst)

Saved: /content/drive/MyDrive/trialsclean.csv
Saved: /content/drive/MyDrive/compliancemetrics.csv


In [12]:
TRIALSCSV = "/content/drive/MyDrive/trialsclean.csv"
COMPLIANCECSV = "/content/drive/MyDrive/compliancemetrics.csv"